<a href="https://colab.research.google.com/github/nourhnnan/sentiment-analysis-project/blob/main/sentimentProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q kagglehub

import kagglehub, os
import pandas as pd

path = kagglehub.dataset_download("yacharki/amazon-reviews-for-sa-binary-negative-positive-csv")
print(path)

for root, dirs, files in os.walk(path):
    print(root, files)

Using Colab cache for faster access to the 'amazon-reviews-for-sa-binary-negative-positive-csv' dataset.
/kaggle/input/amazon-reviews-for-sa-binary-negative-positive-csv
/kaggle/input/amazon-reviews-for-sa-binary-negative-positive-csv ['version_3.2.0.txt']
/kaggle/input/amazon-reviews-for-sa-binary-negative-positive-csv/amazon_review_sa_binary_csv ['readme.txt', 'train.csv', 'test.csv']


In [2]:
import glob
train_path = glob.glob(path + "/**/train.csv", recursive=True)[0]
print(train_path)

print(pd.read_csv(train_path, header=None, nrows=3))

/kaggle/input/amazon-reviews-for-sa-binary-negative-positive-csv/amazon_review_sa_binary_csv/train.csv
             0                                      1  \
0  class_index                           review_title   
1            2         Stuning even for the non-gamer   
2            2  The best soundtrack ever to anything.   

                                                   2  
0                                        review_text  
1  This sound track was beautiful! It paints the ...  
2  I'm reading a lot of reviews saying that this ...  


In [3]:
parts = []
for chunk in pd.read_csv(train_path, chunksize=200_000):
    parts.append(chunk.sample(frac=0.03, random_state=42))

sample = pd.concat(parts, ignore_index=True)
sample["label"] = sample["class_index"].map({1: 0, 2: 1})

print(sample.shape)
print(sample["label"].value_counts(normalize=True))
sample.head()

(108000, 4)
label
1    0.501176
0    0.498824
Name: proportion, dtype: float64


,class_index,review_title,review_text,label
0,2,The Sweet Smell of Christmas,My gradchildren love this book and have scratc...,1
1,2,given the terrible circumstances...,The Beatles were on their last legs during the...,1
2,2,Your Eyes'll Be Glued To This Book!!,"""SHILOH"" is a great book for all ages! Well, n...",1
3,2,Quite good -- if only there were more examples...,"First, I unequivocally recommend this book to ...",1
4,1,Disappointed,I bought 2 of these discs. Both seem like bad ...,0


In [4]:
from google.colab import drive
drive.mount("/content/drive")

sample.to_csv("/content/drive/MyDrive/reviews_sample.csv", index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
print(os.path.exists("/content/drive/MyDrive/reviews_sample.csv"))
print(sample.isnull().sum())
print(sample.duplicated(subset=["review_title", "review_text"]).sum())

True
class_index     0
review_title    8
review_text     0
label           0
dtype: int64
0


In [6]:
import re

sample["review_title"] = sample["review_title"].fillna("")
sample["review"] = sample["review_title"] + " " + sample["review_text"]

def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-z\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

sample["clean_review"] = sample["review"].apply(clean_text)

print(sample[["review", "clean_review"]].head())
print((sample["clean_review"].str.len() == 0).sum())

                                              review  \
0  The Sweet Smell of Christmas My gradchildren l...   
1  given the terrible circumstances... The Beatle...   
2  Your Eyes'll Be Glued To This Book!! "SHILOH" ...   
3  Quite good -- if only there were more examples...   
4  Disappointed I bought 2 of these discs. Both s...   

                                        clean_review  
0  the sweet smell of christmas my gradchildren l...  
1  given the terrible circumstances the beatles w...  
2  your eyes'll be glued to this book shiloh is a...  
3  quite good if only there were more examples fi...  
4  disappointed i bought of these discs both seem...  
0


In [7]:
from sklearn.model_selection import train_test_split

X = sample["clean_review"]
y = sample["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(86400,) (21600,)
label
1    0.501181
0    0.498819
Name: proportion, dtype: float64
label
1    0.501157
0    0.498843
Name: proportion, dtype: float64


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=50000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(X_train_vec.shape)
print(X_test_vec.shape)

(86400, 50000)
(21600, 50000)


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

pred = model.predict(X_test_vec)

print(accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

0.8956481481481482
              precision    recall  f1-score   support

           0       0.89      0.90      0.90     10775
           1       0.90      0.89      0.90     10825

    accuracy                           0.90     21600
   macro avg       0.90      0.90      0.90     21600
weighted avg       0.90      0.90      0.90     21600



In [10]:
reviews = [
    "This product is amazing, I love it",
    "Terrible quality, it broke after one day",
]

cleaned = [clean_text(r) for r in reviews]
vec = vectorizer.transform(cleaned)

print(model.predict(vec))
print(model.predict_proba(vec))

[1 0]
[[4.10635206e-04 9.99589365e-01]
 [9.98491253e-01 1.50874676e-03]]
